## Gold Core Tables

Builds **four Gold tables** using existing Silver outputs.
### Tables Created

- **Gold_Fund_Snapshot**
  - Joins `silver.fund` and `silver.fund_financials` on `fund_id`.
  - `fund_id` is the only shared column, making the join unambiguous.

- **Gold_Fund_Financials**
  - Pass-through from `silver.fund_financials`.

- **Gold_Portfolio_Valuation**
  - Pass-through from `silver.portfolio_valuation`.
  - Includes company, fund, benchmark, position, and valuation fields.

- **Gold_Reconciliation**
  - Based on `silver.reconciliation`.
  - Adds `source_a_numeric` and `source_b_numeric` so Power BI can chart CASH dollar values numerically.
  - Non-numeric REFERENCE currency-code values safely become null through `try_cast`.

### Gold_Fund_Snapshot Source Schemas

**silver.fund**
- `fund_id`
- `fund_name`
- `vintage_year`
- `fund_size_usd`

**silver.fund_financials**

- `fund_id`
- `total_commitments`
- `contributions`
- `uncalled_capital`
- `invested_capital`
- `portfolio_value`
- `available_cash_internal`
- `available_cash_external`
- `estimated_nav`
- `calculated_at`

### Gold_Portfolio_Valuation

Source: `silver.portfolio_valuation`

Includes:

- `company_id`
- `company_name`
- `fund_id`
- `benchmark_ticker`
- `latest_close`
- `quantity`
- `market_value`
- `has_benchmark`
- `has_position`

### Gold Tables in This Notebook

| Gold Table | Source | Logic |
|---|---|---|
| `Gold_Fund_Snapshot` | `silver.fund` + `silver.fund_financials` | Join |
| `Gold_Fund_Financials` | `silver.fund_financials` | Pass-through |
| `Gold_Portfolio_Valuation` | `silver.portfolio_valuation` | Pass-through |
| `Gold_Reconciliation` | `silver.reconciliation` | Add numeric source columns |


In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F

## A. `Gold_Fund_Snapshot`

`silver.fund` (fund_id, fund_name, vintage_year, fund_size_usd) left-joined with `silver.fund_financials` on `fund_id` - the only shared column, so no ambiguous-reference risk.

In [0]:
fund_df = spark.table(silver_table("fund"))
fund_financials_df = spark.table(silver_table("fund_financials"))

fund_snapshot_df = (
    fund_df
    .join(fund_financials_df, on="fund_id", how="left")
    .withColumn("snapshot_generated_at", F.current_timestamp())
)

write_gold(fund_snapshot_df, "fund_snapshot")
print(f"Gold_Fund_Snapshot row count: {fund_snapshot_df.count()}")
fund_snapshot_df.show(truncate=False)

Gold_Fund_Snapshot row count: 5
+--------+-------------------------+------------+-------------+------------------------------------+--------------------------+-----------------+--------------------+----------------+------------------+---------------+-----------------------+-----------------------+--------------------+--------------------------+--------------------------+
|fund_id |fund_name                |vintage_year|fund_size_usd|_run_id                             |_ingested_at              |total_commitments|contributions       |uncalled_capital|invested_capital  |portfolio_value|available_cash_internal|available_cash_external|estimated_nav       |calculated_at             |snapshot_generated_at     |
+--------+-------------------------+------------+-------------+------------------------------------+--------------------------+-----------------+--------------------+----------------+------------------+---------------+-----------------------+-----------------------+------------------

## B. `Gold_Fund_Financials`

Direct pass-through of `silver.fund_financials` - already fund-grain, Gold just gives it a stable, business-facing name and location.

In [0]:
fund_financials_gold_df = (
    spark.table(silver_table("fund_financials"))
    .withColumn("gold_loaded_at", F.current_timestamp())
)

write_gold(fund_financials_gold_df, "fund_financials")
print(f"Gold_Fund_Financials row count: {fund_financials_gold_df.count()}")
fund_financials_gold_df.show(truncate=False)

Gold_Fund_Financials row count: 5
+--------+-----------------+--------------------+----------------+------------------+---------------+-----------------------+-----------------------+--------------------+--------------------------+--------------------------+
|fund_id |total_commitments|contributions       |uncalled_capital|invested_capital  |portfolio_value|available_cash_internal|available_cash_external|estimated_nav       |calculated_at             |gold_loaded_at            |
+--------+-----------------+--------------------+----------------+------------------+---------------+-----------------------+-----------------------+--------------------+--------------------------+--------------------------+
|FUND_004|1.91155002E8     |2.485377029E7       |1.6452784902E8  |2139281.6999999997|0.0            |2.271448859E7          |0.0                    |2.271448859E7       |2026-09-23 03:34:11.055874|2026-09-23 05:22:21.253522|
|FUND_002|8.1854831E7      |1.3992916270000003E7|8.041980029E7   |

## C. `Gold_Portfolio_Valuation`

Direct pass-through of `silver.portfolio_valuation` - already company-grain with quantity/price/market_value/has_position/has_benchmark computed in `13`. Same treatment as Fund_Financials above.

In [0]:
portfolio_valuation_gold_df = (
    spark.table(silver_table("portfolio_valuation"))
    .withColumn("gold_loaded_at", F.current_timestamp())
)

write_gold(portfolio_valuation_gold_df, "portfolio_valuation")
print(f"Gold_Portfolio_Valuation row count: {portfolio_valuation_gold_df.count()}")
portfolio_valuation_gold_df.show(truncate=False)

Gold_Portfolio_Valuation row count: 30
+----------+-----------------------------+--------+----------------+------------+-------------+--------+------------+------------+--------------------------+
|company_id|company_name                 |fund_id |benchmark_ticker|latest_close|has_benchmark|quantity|market_value|has_position|gold_loaded_at            |
+----------+-----------------------------+--------+----------------+------------+-------------+--------+------------+------------+--------------------------+
|PORT_0001 |Henson-Johnston              |FUND_005|UNH             |377.54      |true         |NULL    |NULL        |false       |2026-09-23 05:22:25.401227|
|PORT_0002 |Ramos, Carr and Cook         |FUND_002|OR.PA           |383.65      |true         |NULL    |NULL        |false       |2026-09-23 05:22:25.401227|
|PORT_0003 |Lang-Anderson                |FUND_001|MC.PA           |406.9       |true         |2580.0  |1049802.0   |true        |2026-09-23 05:22:25.401227|
|PORT_0004 |L

## D. `Gold_Reconciliation`

`silver.reconciliation` extended with numeric companion columns for Power BI. `source_a_value`/`source_b_value` stay as strings (audit trail, and REFERENCE needs to hold currency codes in the same columns CASH uses for dollar amounts - one shared schema across all 3 recon types was the reason they are strings to begin with). The `*_numeric` columns below are a parallel, chart-ready view: populated only where the underlying value genuinely is a number (CASH dollar amounts, POSITION quantities), NULL everywhere else (REFERENCE currency codes, and any NULL source values). `try_cast` returns NULL on a failed cast instead of erroring, so it is safe to apply across all three recon_types in one pass.

In [0]:
reconciliation_gold_df = (
    spark.table(silver_table("reconciliation"))
    .withColumn("source_a_numeric", F.expr("try_cast(source_a_value AS double)"))
    .withColumn("source_b_numeric", F.expr("try_cast(source_b_value AS double)"))
    .withColumn("gold_loaded_at", F.current_timestamp())
)

write_gold(reconciliation_gold_df, "reconciliation")
print(f"Gold_Reconciliation row count: {reconciliation_gold_df.count()}")

print("Numeric cast coverage by recon_type (sanity check):")
(
    reconciliation_gold_df
    .groupBy("recon_type")
    .agg(
        F.count("*").alias("total_rows"),
        F.count("source_a_numeric").alias("a_numeric_populated"),
        F.count("source_b_numeric").alias("b_numeric_populated"),
    )
    .show()
)

reconciliation_gold_df.show(truncate=False)

Gold_Reconciliation row count: 55
Numeric cast coverage by recon_type (sanity check):
+----------+----------+-------------------+-------------------+
|recon_type|total_rows|a_numeric_populated|b_numeric_populated|
+----------+----------+-------------------+-------------------+
|  POSITION|        32|                  2|                  3|
| REFERENCE|        18|                  0|                  0|
|      CASH|         5|                  5|                  5|
+----------+----------+-------------------+-------------------+

+----------------------------------------------------------------+----------+-------------+--------+---------+--------------+--------------+----------+----------------+-----------------------------+--------------------------+----------------+----------------+--------------------------+
|recon_id                                                        |recon_type|business_date|fund_id |entity_id|source_a_value|source_b_value|difference|status          |break_reas

### Summary

In [0]:
print("=== 15_gold_core complete ===")
for t in ["fund_snapshot", "fund_financials", "portfolio_valuation", "reconciliation"]:
    cnt = spark.table(gold_table(t)).count()
    print(f"  {gold_table(t)}: {cnt} rows")

=== 15_gold_core complete ===
  dbw_pe_platform.gold.fund_snapshot: 5 rows
  dbw_pe_platform.gold.fund_financials: 5 rows
  dbw_pe_platform.gold.portfolio_valuation: 30 rows
  dbw_pe_platform.gold.reconciliation: 55 rows
